# MCP Text2SQL Project Tests

This notebook runs practical checks across guard, logger, DB, tools, API routes, and optional OpenAI call.

In [ ]:
import json
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv
from fastapi.testclient import TestClient
from sqlalchemy import text

import logger
import db_connection
import guard
import tools
import mcp_server

# Make sure .env is loaded for notebook tests
load_dotenv()

RESULTS = []

def add_result(name: str, status: str, details: str = ""):
    RESULTS.append({"test": name, "status": status, "details": details})

def mark_ok(name: str, details: str = ""):
    add_result(name, "PASS", details)

def mark_fail(name: str, details: str = ""):
    add_result(name, "FAIL", details)

def mark_skip(name: str, details: str = ""):
    add_result(name, "SKIP", details)

print("Imports and test harness ready")

In [ ]:
# guard.py tests
try:
    out = guard.sanitize_and_validate_sql("SELECT name FROM sys.tables")
    if "TOP" in out.upper():
        mark_ok("guard_adds_top", out)
    else:
        mark_fail("guard_adds_top", f"Unexpected output: {out}")
except Exception as e:
    mark_fail("guard_adds_top", repr(e))

try:
    bad_q = "SELECT * FROM a JOIN b ON a.id=b.id JOIN c ON b.id=c.id JOIN d ON c.id=d.id"
    guard.sanitize_and_validate_sql(bad_q)
    mark_fail("guard_limits_complexity", "Expected ValueError but query passed")
except ValueError as e:
    if "max table limit" in str(e).lower() or "max complexity" in str(e).lower():
        mark_ok("guard_limits_complexity", str(e))
    else:
        mark_fail("guard_limits_complexity", f"Wrong error: {e}")
except Exception as e:
    mark_fail("guard_limits_complexity", repr(e))

print("Guard tests completed")

In [ ]:
# logger.py tests
try:
    trace_id = f"nb-log-{uuid.uuid4()}"
    logger.log_event({"trace_id": trace_id, "event_type": "tool_ok", "tool_name": "notebook_logger_test", "sql_preview": "SELECT * FROM dbo.Users"})

    log_path = Path(os.getenv("LOG_PATH", "logs/events.jsonl"))
    if not log_path.exists():
        mark_fail("logger_writes_jsonl", f"Missing log file: {log_path}")
    else:
        last_line = log_path.read_text(encoding="utf-8").strip().splitlines()[-1]
        rec = json.loads(last_line)
        if rec.get("trace_id") == trace_id and rec.get("event_type") == "tool_ok":
            mark_ok("logger_writes_jsonl", f"line={last_line[:120]}")
        else:
            mark_fail("logger_writes_jsonl", f"Unexpected log record: {rec}")
except Exception as e:
    mark_fail("logger_writes_jsonl", repr(e))

print("Logger test completed")

In [ ]:
# DB + tools tests (depend on reachable SQL Server)
db_trace = f"nb-db-{uuid.uuid4()}"

try:
    with db_connection.get_connection() as conn:
        row = conn.execute(text("SELECT 1 AS ok")).first()
    if row and int(row[0]) == 1:
        mark_ok("db_connection", "SELECT 1 succeeded")
    else:
        mark_fail("db_connection", f"Unexpected row: {row}")
except Exception as e:
    mark_skip("db_connection", f"DB not reachable/configured: {e}")

if any(r["test"] == "db_connection" and r["status"] == "PASS" for r in RESULTS):
    try:
        schema = tools.get_schema(db_trace)
        mark_ok("tool_get_schema", f"tables={len(schema.get('tables', {}))}, views={len(schema.get('views', {}))}, rel={len(schema.get('relationships', []))}")
    except Exception as e:
        mark_fail("tool_get_schema", repr(e))

    try:
        q_res = tools.execute_readonly_sql(db_trace, "SELECT TOP (5) name FROM sys.tables")
        mark_ok("tool_execute_readonly_sql", f"rows={q_res.get('row_count')}")
    except Exception as e:
        mark_fail("tool_execute_readonly_sql", repr(e))

    try:
        first_table = next(iter(schema.get("tables", {})), None)
        if not first_table:
            raise ValueError("No base tables found for preview test")
        schema_name, table_name = first_table.split(".", 1)
        p_res = tools.preview_table(db_trace, table_name=table_name, schema_name=schema_name)
        mark_ok("tool_preview_table", f"table={schema_name}.{table_name}, rows={p_res.get('row_count')}")
    except Exception as e:
        mark_skip("tool_preview_table", f"Preview unavailable: {e}")
else:
    mark_skip("tool_get_schema", "Skipped because DB connection failed")
    mark_skip("tool_execute_readonly_sql", "Skipped because DB connection failed")
    mark_skip("tool_preview_table", "Skipped because DB connection failed")

# explain_reasoning is local-only and should always work
try:
    er = tools.explain_reasoning(db_trace, "List tables", ["sys.tables"], "SELECT TOP (5) name FROM sys.tables")
    mark_ok("tool_explain_reasoning", er.get("limit_policy", ""))
except Exception as e:
    mark_fail("tool_explain_reasoning", repr(e))

print("DB/tools tests completed")

In [ ]:
# FastAPI route tests (in-process, no external server needed)
try:
    client = TestClient(mcp_server.app)

    h = client.get("/health")
    if h.status_code == 200:
        mark_ok("api_health", h.text)
    else:
        mark_fail("api_health", f"status={h.status_code}, body={h.text}")

    headers = {}
    key = os.getenv("MCP_API_KEY", "").strip()
    if key:
        headers["x-api-key"] = key

    er_body = {"question": "q", "chosen_tables": ["dbo.t"], "sql": "SELECT 1"}
    er_resp = client.post("/tools/explain_reasoning", headers=headers, json=er_body)
    if er_resp.status_code == 200:
        mark_ok("api_explain_reasoning", er_resp.text[:120])
    else:
        mark_fail("api_explain_reasoning", f"status={er_resp.status_code}, body={er_resp.text}")

    list_payload = {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
    list_resp = client.post("/mcp", headers=headers, json=list_payload)
    if list_resp.status_code == 200 and "result" in list_resp.json():
        mark_ok("api_mcp_tools_list", "tools/list returned result")
    else:
        mark_fail("api_mcp_tools_list", f"status={list_resp.status_code}, body={list_resp.text}")

except Exception as e:
    mark_fail("api_tests", repr(e))

print("API tests completed")

In [ ]:
# Optional OpenAI test (uses OPENAI_API_KEY from .env)
try:
    from langchain_openai import ChatOpenAI

    api_key = os.getenv("OPENAI_API_KEY", "").strip()
    if not api_key:
        mark_skip("openai_call", "OPENAI_API_KEY not set")
    else:
        model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
        llm = ChatOpenAI(model=model, api_key=api_key, temperature=0)
        msg = llm.invoke("Reply with exactly: OPENAI_OK")
        content = getattr(msg, "content", str(msg))
        if "OPENAI_OK" in str(content):
            mark_ok("openai_call", str(content))
        else:
            mark_fail("openai_call", f"Unexpected response: {content}")
except Exception as e:
    mark_skip("openai_call", f"Skipped due to runtime/network/provider issue: {e}")

print("OpenAI test completed")

## Direct Tool Calls (One Tool Per Cell)

Run these cells to see each tool response directly.

In [ ]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
schema_result = tools.get_schema(direct_trace)
print("tables:", len(schema_result.get("tables", {})))
print("views:", len(schema_result.get("views", {})))
print("relationships:", len(schema_result.get("relationships", [])))
list(schema_result.get("tables", {}).keys())[:5]

In [ ]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
sql_result = tools.execute_readonly_sql(direct_trace, "SELECT TOP (5) name FROM sys.tables")
print("sanitized sql:", sql_result.get("sql"))
print("row_count:", sql_result.get("row_count"))
sql_result.get("rows", [])

In [ ]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
schema_for_preview = tools.get_schema(direct_trace)
first_table = next(iter(schema_for_preview.get("tables", {})), None)
if not first_table:
    raise ValueError("No base table found for preview")
schema_name, table_name = first_table.split(".", 1)
preview_result = tools.preview_table(direct_trace, table_name=table_name, schema_name=schema_name)
print("table:", preview_result.get("table"))
print("row_count:", preview_result.get("row_count"))
preview_result.get("rows", [])

In [ ]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
explain_result = tools.explain_reasoning(
    direct_trace,
    question="Show me table names",
    chosen_tables=["sys.tables"],
    sql="SELECT TOP (5) name FROM sys.tables"
)
explain_result

## LLM Prompt -> MCP Tool Calls (LangChain)

This demo sends your prompt to the model. The model can decide to call MCP tools via `/mcp`.

In [ ]:
import requests
from requests.exceptions import RequestException
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

MCP_BASE_URL = os.getenv("MCP_BASE_URL", "http://127.0.0.1:8000")
MCP_API_KEY = os.getenv("MCP_API_KEY", "").strip()
local_client = None

SYSTEM_PROMPT = (
    "You are a SQL assistant. Use MCP tools when useful. "
    "You MUST always return a non-empty final plain-text answer with exactly 3 bullet points: "
    "what you found, key numbers, suggested next query."
)

def _mcp_headers():
    h = {"Content-Type": "application/json"}
    if MCP_API_KEY:
        h["x-api-key"] = MCP_API_KEY
    return h

def _try_enable_local_fallback():
    global local_client
    if local_client is None:
        local_client = TestClient(mcp_server.app)

def mcp_rpc(method: str, params: dict | None = None):
    payload = {"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": method}
    if params is not None:
        payload["params"] = params

    try:
        resp = requests.post(f"{MCP_BASE_URL}/mcp", headers=_mcp_headers(), json=payload, timeout=30)
        resp.raise_for_status()
        body = resp.json()
    except Exception:
        _try_enable_local_fallback()
        resp = local_client.post("/mcp", headers=_mcp_headers(), json=payload)
        body = resp.json()

    if "error" in body:
        raise RuntimeError(f"MCP error: {body['error']}")
    return body

@tool
def mcp_get_schema() -> dict:
    """Get DB tables, views, and relationships from MCP."""
    return mcp_rpc("tools/call", {"name": "get_schema", "arguments": {}})["result"]

@tool
def mcp_execute_readonly_sql(sql: str) -> dict:
    """Execute read-only SQL through MCP."""
    return mcp_rpc("tools/call", {"name": "execute_readonly_sql", "arguments": {"sql": sql}})["result"]

@tool
def mcp_preview_table(table_name: str, schema_name: str = "dbo") -> dict:
    """Preview first 5 rows of a table through MCP."""
    return mcp_rpc("tools/call", {"name": "preview_table", "arguments": {"table_name": table_name, "schema_name": schema_name}})["result"]

@tool
def mcp_explain_reasoning(question: str, chosen_tables: list[str], sql: str) -> dict:
    """Explain SQL reasoning through MCP."""
    return mcp_rpc("tools/call", {"name": "explain_reasoning", "arguments": {"question": question, "chosen_tables": chosen_tables, "sql": sql}})["result"]

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    raise ValueError("OPENAI_API_KEY is required in .env")

model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=model_name, api_key=api_key, temperature=0)
llm_with_tools = llm.bind_tools([mcp_get_schema, mcp_execute_readonly_sql, mcp_preview_table, mcp_explain_reasoning])

print("LangChain + MCP ready")
print("MCP_BASE_URL:", MCP_BASE_URL)
print("Fallback mode enabled automatically if HTTP MCP is not running")

In [ ]:
# Put your own prompt here
user_prompt = """
You are a SQL assistant.
Use MCP tools if needed.
You MUST always return a non-empty final plain-text answer with exactly 3 bullet points:
- what you found
- key numbers
- suggested next query
Question: Find 5 employee names from the warehouse DB and explain briefly.
"""

messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=user_prompt)]
tool_map = {
    "mcp_get_schema": mcp_get_schema,
    "mcp_execute_readonly_sql": mcp_execute_readonly_sql,
    "mcp_preview_table": mcp_preview_table,
    "mcp_explain_reasoning": mcp_explain_reasoning,
}

tool_outputs = []
final_text = ""

for step in range(1, 6):
    reply = llm_with_tools.invoke(messages)
    messages.append(reply)
    print(f"\nStep {step} model content:", repr(reply.content))
    print("tool_calls:", reply.tool_calls)

    if not reply.tool_calls:
        final_text = (reply.content or "").strip()
        if final_text:
            break
        messages.append(HumanMessage(content="Return now exactly 3 plain-text bullet points. Do not leave empty."))
        continue

    for tc in reply.tool_calls:
        name = tc["name"]
        args = tc.get("args", {})
        if name not in tool_map:
            tool_output = {"error": f"Unknown tool from model: {name}"}
        else:
            tool_output = tool_map[name].invoke(args)
        tool_outputs.append({"name": name, "args": args, "output": tool_output})
        print(f"Tool {name} args={args}")
        print("Tool output preview:", repr(str(tool_output)[:500]))
        messages.append(ToolMessage(content=json.dumps(tool_output, ensure_ascii=False), tool_call_id=tc["id"]))

if not final_text:
    last_sql = None
    last_rows = None
    for rec in reversed(tool_outputs):
        out = rec.get("output", {})
        if isinstance(out, dict):
            if last_sql is None and out.get("sql"):
                last_sql = out.get("sql")
            if last_rows is None and out.get("row_count") is not None:
                last_rows = out.get("row_count")
    fallback_rows = last_rows if last_rows is not None else "n/a"
    fallback_sql = last_sql if last_sql else "SELECT TOP (10) * FROM dbo.Dim_Employees"
    final_text = (
        "- what you found: Schema and/or query results were retrieved through MCP tools.\n"
        f"- key numbers: tool_calls={len(tool_outputs)}, row_count={fallback_rows}.\n"
        f"- suggested next query: {fallback_sql}"
    )

print("\nFinal model response after MCP tool results:")
print(final_text)


## OpenAI SDK + MCP (Below LangChain)\n\nThis uses OpenAI SDK directly with MCP tool type (no LangChain tool wrappers).

In [ ]:
from openai import OpenAI\n\nOPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()\nif not OPENAI_API_KEY:\n    raise ValueError("OPENAI_API_KEY is required in .env")\n\nSDK_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")\nSDK_MCP_BASE_URL = os.getenv("MCP_BASE_URL", "http://127.0.0.1:8000").rstrip("/")\nSDK_MCP_API_KEY = os.getenv("MCP_API_KEY", "").strip()\n\nsdk_client = OpenAI(api_key=OPENAI_API_KEY)\n\nsdk_mcp_tool = {\n    "type": "mcp",\n    "server_label": "local_sql_mcp",\n    "server_url": f"{SDK_MCP_BASE_URL}/mcp",\n    "require_approval": "never"\n}\nif SDK_MCP_API_KEY:\n    sdk_mcp_tool["headers"] = {"x-api-key": SDK_MCP_API_KEY}\n\nprint("OpenAI SDK MCP setup ready")\nprint("MODEL:", SDK_MODEL)\nprint("MCP URL:", sdk_mcp_tool["server_url"])

In [ ]:
# Put your own prompt here\nsdk_prompt = "Find 5 employee names from the warehouse DB and explain briefly in 3 bullet points."\n\nsdk_response = sdk_client.responses.create(\n    model=SDK_MODEL,\n    input=sdk_prompt,\n    tools=[sdk_mcp_tool]\n)\n\nprint("Response ID:", sdk_response.id)\nprint("\\nFinal text:")\nprint(getattr(sdk_response, "output_text", ""))\n\nprint("\\nOutput items (inspect MCP behavior):")\nfor idx, item in enumerate(getattr(sdk_response, "output", []) or []):\n    item_type = getattr(item, "type", None) if not isinstance(item, dict) else item.get("type")\n    print(f"[{idx}] type={item_type}")\n    print(item)

In [ ]:
from collections import Counter

counts = Counter(r['status'] for r in RESULTS)
print("Test Summary:")
print(dict(counts))

for row in RESULTS:
    print(f"- [{row['status']}] {row['test']}: {row['details']}")

RESULTS